# Activation patching -- do layers 19-26 *cause* the flip? (Colab, GPU)

The logit lens showed *where* flipping items differ (layers 19-26). This tests *cause*.

For each flipping item: run the **source** model (peak, where it answers with the context) and keep its
residual stream after every layer. Then run the **target** model (trough, where it answers with its own
answer), but at one layer `l` replace the target's residual stream (all positions) with the source's.
Every layer after `l` still runs with the **target's** weights.

    R(l) = (D_patched - D_target) / (D_source - D_target)       0 = no effect, 1 = fully restored

`D` = logit(first token of context answer) - logit(first token of own answer) at the model's output,
on the generation prompt -- the same measure as the logit lens (v2).

**Reading the sweep.** R(l) is high only once layer `l` is past the target's "converter":
- change lives in layers 19-26 -> R ~ 0 up to layer 18, rises to ~ 1 across 19-26
- change happens earlier (invisible to the lens) -> R already high by layer 18
- change lives in the last layers -> R stays low until ~ 27+

**Pre-registered prediction.** The *restoration layer* -- first layer where the median R across flipping
items reaches 0.5 -- lies in 19-26 for all three comparisons (tulu drop, alpaca drop, tulu recovery).
It fails if it is <= 18 or >= 27 for any of them.

Lens layer `l` = output of block `model.model.layers[l - 1]`; the no-op check below verifies this.

In [ ]:
import os

GITHUB_REPO_URL = "https://github.com/GIRIAYUSH/context-parametric-inversion-research.git"
REPO_DIR = "/content/context-parametric-inversion-research"
BRANCH = "dev"

if not os.path.isdir(REPO_DIR):
    !git clone --branch {BRANCH} {GITHUB_REPO_URL} {REPO_DIR}
%cd {REPO_DIR}
!git fetch origin {BRANCH} && git reset --hard origin/{BRANCH}
!git log --oneline -3

!pip install -q peft accelerate

from google.colab import drive
drive.mount('/content/drive')

RESULTS = "/content/drive/MyDrive/CPI-Analysis-Results"
LENS_DIR = f"{RESULTS}/logit-lens-v2"            # input: used to pick the flipping items
OUT_DIR = f"{RESULTS}/activation-patching"        # output, persistent
os.makedirs(OUT_DIR, exist_ok=True)
assert os.access(OUT_DIR, os.W_OK), "Drive output dir not writable"
assert os.path.isdir(LENS_DIR), f"missing {LENS_DIR} -- run logit_lens.ipynb first"
print("writing to", OUT_DIR)

In [ ]:
import json, yaml, torch, sys, shutil, time, glob
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
from huggingface_hub import login
from google.colab import userdata

login(token=userdata.get("HF_TOKEN"))

sys.path.insert(0, "src/evaluation")
import eval as cpi_eval

cfg = yaml.safe_load(open("src/mechanistic-analysis/checkpoints.yaml"))
tok = AutoTokenizer.from_pretrained(cfg["base_model"])
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# Llama-2-7b-hf ships no chat template; must match run_sft.py's training format exactly.
tok.chat_template = (
    "{{ bos_token }} "
    "{% for message in messages %}"
    "{% if message['role'] == 'system' %}"
    "{{ '<|system|>\n' + message['content'].strip() + '\n' }}"
    "{% elif message['role'] == 'user' %}"
    "{{ '<|user|>\n' + message['content'].strip() + '\n' }}"
    "{% elif message['role'] == 'assistant' %}"
    "{{ '<|assistant|>\n' + message['content'].strip() + eos_token + '\n' }}"
    "{% endif %}"
    "{% endfor %}"
    "{% if add_generation_prompt %}{{ '<|assistant|>\n' }}{% endif %}"
)
ref = json.load(open("results/cpi-results/tulu-results/checkpoint_step3150_metrics.json", encoding="utf-8"))
row = next(r for r in ref["per_item"] if r["item_id"] == "cap_0013")
assert tok.decode(cpi_eval.build_gen_prompt_ids(tok, row["question"], row["context"], True),
                  skip_special_tokens=False) == row["gen_prompt_text"], "generation prompt mismatch"
print("prompt verified")

items = {it["item_id"]: it for it in cpi_eval.load_items("dataset/conflict_eval_unified.json")}
base_model = AutoModelForCausalLM.from_pretrained(
    cfg["base_model"], torch_dtype=torch.bfloat16, device_map="cuda").eval()
print(len(items), "items; base model loaded")

In [ ]:
# --- same measure as the logit lens (v2) ---
NEWLINE = tok("\n", add_special_tokens=False).input_ids[-1]

def first_token(answer):
    ids = tok("\n" + answer.strip(), add_special_tokens=False).input_ids
    return ids[ids.index(NEWLINE) + 1]

def prompt_ids(item):
    return cpi_eval.build_gen_prompt_ids(tok, item["question"], item["context"], True).unsqueeze(0).to("cuda")

def lean(logits, item):
    # D at the last position: logit(context answer) - logit(own answer)
    z = logits[0, -1].float()
    return (z[first_token(item["counterfactual_answer"])] - z[first_token(item["parametric_answer"])]).item()

# --- the patch: a forward hook that swaps one block's output for a stored residual stream ---
def replace_output(new_hidden):
    def hook(module, inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output   # older transformers return a tuple
        if hidden.shape[1] != new_hidden.shape[1]:
            return output               # a generation step for a newly generated token: leave it alone
        return (new_hidden,) + tuple(output[1:]) if isinstance(output, tuple) else new_hidden
    return hook

@torch.no_grad()
def run(model, ids, patch_layer=None, patch_value=None):
    # Forward pass; if patch_layer is given, the residual stream after that lens layer is replaced.
    handle = None
    if patch_layer is not None:
        block = model.get_base_model().model.layers[patch_layer - 1]   # lens layer l = output of block l-1
        handle = block.register_forward_hook(replace_output(patch_value))
    try:
        return model(ids, output_hidden_states=True)
    finally:
        if handle is not None:
            handle.remove()

@torch.no_grad()
def generate(model, ids, patch_layer=None, patch_value=None):
    # Greedy generation, same settings as the eval. The patch replaces the prompt's residual stream at
    # patch_layer; tokens generated afterwards are computed normally (by the current adapter).
    handle = None
    if patch_layer is not None:
        block = model.get_base_model().model.layers[patch_layer - 1]
        handle = block.register_forward_hook(replace_output(patch_value))
    try:
        out = model.generate(input_ids=ids, max_new_tokens=cpi_eval.MAX_NEW_TOKENS, do_sample=False, use_cache=True,
                             pad_token_id=tok.pad_token_id or tok.eos_token_id)
    finally:
        if handle is not None:
            handle.remove()
    return tok.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

def answer_label(text, item):
    # the eval's own generation classifier: CTX / PAR / OTHER
    return cpi_eval.method_ordered(text, item["parametric_answer"], item["counterfactual_answer"])["label"]

# --- two adapters held at once; switching between them is instant ---
STAGE = "/content/_ckpt"

def load_pair(source_path, target_path):
    global base_model
    if isinstance(base_model, PeftModel):
        base_model = base_model.unload()
    shutil.rmtree(STAGE, ignore_errors=True)
    shutil.copytree(source_path, f"{STAGE}/source")        # local copies: Drive reads are slow
    shutil.copytree(target_path, f"{STAGE}/target")
    base_model = PeftModel.from_pretrained(base_model, f"{STAGE}/source", adapter_name="source")
    base_model.load_adapter(f"{STAGE}/target", adapter_name="target")
    return base_model.eval()

In [ ]:
# Which comparisons, and which items: flipping capitals taken from the logit-lens v2 results
# (final-layer lean > 0 in the source checkpoint, < 0 in the target), so both experiments use the same items.
run_dir = lambda r: cfg["runs"][r]["drive_checkpoint_dir"]
step = lambda r, phase: int(cfg["runs"][r]["phases"][phase].split("-")[1])

COMPARISONS = [  # (run, source phase, target phase)
    ("tulu", "peak", "trough"),
    ("alpaca", "peak", "trough"),
    ("tulu", "recovery", "trough"),      # recovery answers with the context again, like the peak
]

def lens_final(r, phase):
    d = json.load(open(f"{LENS_DIR}/{r}_seed0_{phase}_step{step(r, phase)}.json"))
    return {i: v["D"][-1] for i, v in d["curves"].items() if not v["same_first_token"]}

JOBS = []
for run_name, src_phase, tgt_phase in COMPARISONS:     # (don't name these run/src: that would shadow run())
    S, T = lens_final(run_name, src_phase), lens_final(run_name, tgt_phase)
    flips = [i for i in S if i.startswith("cap_") and S[i] > 0 and T.get(i, 0) < 0]
    JOBS.append({"run": run_name, "source": src_phase, "target": tgt_phase, "items": flips,
                 "source_path": f"{run_dir(run_name)}/checkpoint-{step(run_name, src_phase)}",
                 "target_path": f"{run_dir(run_name)}/checkpoint-{step(run_name, tgt_phase)}",
                 "lens_S": S, "lens_T": T})
    print(f"{run_name} {src_phase}->{tgt_phase}: {len(flips)} flipping capitals")

## Sanity checks (on the tulu peak -> trough pair, before the full run)
1. **Clean runs reproduce the logit lens** -- `D` without patching equals the lens file's final value.
2. **No-op patch** -- patching the target with its *own* residual stream must leave `D` unchanged. This also
   proves lens layer `l` really is the output of block `l-1` (an off-by-one would change `D`).
3. **Boundaries** -- patching at layer 1 should restore little (LoRA barely changed the first block); patching
   at layer 31 (only the last block still runs with target weights) should restore most of the way.

In [ ]:
job = JOBS[0]
model = load_pair(job["source_path"], job["target_path"])
check = job["items"][:8]

diffs, noop, r1, r31 = [], [], [], []
for i in check:
    it, ids = items[i], prompt_ids(items[i])
    model.set_adapter("source"); src = run(model, ids)
    model.set_adapter("target"); tgt = run(model, ids)
    D_s, D_t = lean(src.logits, it), lean(tgt.logits, it)
    diffs += [abs(D_s - job["lens_S"][i]), abs(D_t - job["lens_T"][i])]
    for l in [1, 10, 19, 31]:
        noop.append(abs(lean(run(model, ids, l, tgt.hidden_states[l]).logits, it) - D_t))
    r1.append((lean(run(model, ids, 1, src.hidden_states[1]).logits, it) - D_t) / (D_s - D_t))
    r31.append((lean(run(model, ids, 31, src.hidden_states[31]).logits, it) - D_t) / (D_s - D_t))

print(f"1. clean D vs lens file: max difference {max(diffs):.3f}   (want < 0.1)")
print(f"2. no-op patch: max change in D {max(noop):.4f}           (want ~0)")
print(f"3. restoration at layer 1: median {np.median(r1):+.2f}   at layer 31: median {np.median(r31):+.2f}   (want ~0 and near 1)")

In [ ]:
# Full run: every flipping item, every layer 1..31. Resumable: finished comparisons are skipped.
LAYERS = list(range(1, 32))

for job in JOBS:
    name = f"{job['run']}_seed0_{job['source']}_to_{job['target']}"
    out_file = f"{OUT_DIR}/{name}.json"
    if os.path.exists(out_file):
        print("skip (done):", out_file)
        continue
    t0 = time.time()
    model = load_pair(job["source_path"], job["target_path"])
    rows = {}
    for i in job["items"]:
        it, ids = items[i], prompt_ids(items[i])
        model.set_adapter("source"); src = run(model, ids)
        model.set_adapter("target"); tgt = run(model, ids)
        rows[i] = {"D_source": lean(src.logits, it), "D_target": lean(tgt.logits, it),
                   "D_patched": [lean(run(model, ids, l, src.hidden_states[l]).logits, it) for l in LAYERS]}
    json.dump({**{k: job[k] for k in ["run", "source", "target"]}, "layers": LAYERS, "items": rows},
              open(out_file, "w"))
    print(f"{name}: {len(rows)} items in {time.time() - t0:.0f}s")

shutil.rmtree(STAGE, ignore_errors=True)

## Results
Per comparison: median restoration R(l) across flipping items, the share of items whose answer is back to
the context answer (`D_patched > 0`), and the restoration layer (median R first reaches 0.5).

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
ax.axvspan(19, 26, color="gold", alpha=0.15, label="layers 19-26")
verdicts = []
for f in sorted(f for f in glob.glob(f"{OUT_DIR}/*.json") if not f.endswith(("_generation.json", "_necessity.json"))):
    d = json.load(open(f))
    L = np.array(d["layers"])
    R = np.array([(np.array(v["D_patched"]) - v["D_target"]) / (v["D_source"] - v["D_target"]) for v in d["items"].values()])
    back = np.array([np.array(v["D_patched"]) > 0 for v in d["items"].values()])
    med = np.median(R, axis=0)
    q25, q75 = np.percentile(R, [25, 75], axis=0)
    restore = int(L[np.argmax(med >= 0.5)]) if (med >= 0.5).any() else None
    name = f"{d['run']} {d['source']}->{d['target']} (n={len(R)})"
    line, = ax.plot(L, med, linewidth=2, label=name)
    ax.fill_between(L, q25, q75, color=line.get_color(), alpha=0.15)

    print(f"\n{name}: restoration layer = {restore}")
    for l in [16, 18, 19, 20, 22, 24, 26, 28, 31]:
        k = list(L).index(l)
        print(f"   layer {l:2d}: median R {med[k]:+.2f}   back to context answer {back[:, k].mean():.0%}")
    verdicts.append(restore is not None and 19 <= restore <= 26)

ax.axhline(0, color="gray", linewidth=0.8); ax.axhline(1, color="gray", linewidth=0.8, linestyle=":")
ax.set_xlabel("patched layer"); ax.set_ylabel("restoration R (median, IQR band)")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout(); plt.show()

print("\nPre-registered prediction (restoration layer in 19-26 for every comparison):",
      "HOLDS" if verdicts and all(verdicts) else "FAILS")

# Causal check by generation
The logit measure only reads the first-token lean. Here the patched trough model actually **generates** an
answer (greedy, same length as the eval), classified with the eval's own `method_ordered` (CTX / PAR / OTHER).

Conditions per flipping item: unpatched target (baseline, should say its own answer), unpatched source
(should say the context answer), and the target patched at layers 2, 4, ..., 16, 19, 22, 26, 31.

**Pre-registered prediction.** The first patched layer at which >= 50% of flipping items *generate the
context answer* lies in 19-26, for all three comparisons.

In [ ]:
# Sanity checks for generation, on the tulu peak -> trough pair.
#  a. unpatched target generation matches the eval's stored generation at that checkpoint (same label)
#  b. a no-op patch (target's own residual stream) leaves the generated text exactly unchanged
job = JOBS[0]
model = load_pair(job["source_path"], job["target_path"])
stored = {r["item_id"]: r for r in json.load(open(
    f"results/cpi-results/{job['run']}-results/checkpoint_step{step(job['run'], job['target'])}_metrics.json",
    encoding="utf-8"))["per_item"]}

same_label = same_text = 0
check = job["items"][:8]
for i in check:
    it, ids = items[i], prompt_ids(items[i])
    model.set_adapter("target")
    tgt = run(model, ids)
    text = generate(model, ids)
    same_label += answer_label(text, it) == stored[i]["ordered"]["label"]
    same_text += generate(model, ids, 19, tgt.hidden_states[19]) == text
    print(f"{i}: target says {text[:30]!r}  ({answer_label(text, it)})")
print(f"a. label matches stored eval generation: {same_label}/{len(check)}  (want ~all)")
print(f"b. no-op patch leaves text unchanged:     {same_text}/{len(check)}  (want all)")

In [ ]:
# Full generation run. Resumable: finished comparisons are skipped.
GEN_LAYERS = [2, 4, 6, 8, 10, 12, 14, 16, 19, 22, 26, 31]

for job in JOBS:
    name = f"{job['run']}_seed0_{job['source']}_to_{job['target']}_generation"
    out_file = f"{OUT_DIR}/{name}.json"
    if os.path.exists(out_file):
        print("skip (done):", out_file)
        continue
    t0 = time.time()
    model = load_pair(job["source_path"], job["target_path"])
    rows = {}
    for i in job["items"]:
        it, ids = items[i], prompt_ids(items[i])
        model.set_adapter("source")
        src = run(model, ids)
        texts = {"source": generate(model, ids)}
        model.set_adapter("target")
        texts["target"] = generate(model, ids)
        for l in GEN_LAYERS:
            texts[f"layer{l}"] = generate(model, ids, l, src.hidden_states[l])
        rows[i] = {k: {"text": t, "label": answer_label(t, it)} for k, t in texts.items()}
    json.dump({**{k: job[k] for k in ["run", "source", "target"]}, "layers": GEN_LAYERS, "items": rows},
              open(out_file, "w"))
    print(f"{name}: {len(rows)} items in {time.time() - t0:.0f}s")

shutil.rmtree(STAGE, ignore_errors=True)

In [ ]:
# Share of flipping items that GENERATE the context answer, per condition.
gen_verdicts = []
for f in sorted(glob.glob(f"{OUT_DIR}/*_generation.json")):
    d = json.load(open(f))
    rows = list(d["items"].values())
    share = lambda cond: np.mean([r[cond]["label"] == "CTX" for r in rows])
    print(f"\n{d['run']} {d['source']}->{d['target']} (n={len(rows)})")
    print(f"   unpatched target: {share('target'):.0%} say the context answer   (baseline, want ~0%)")
    print(f"   unpatched source: {share('source'):.0%}                           (want ~100%)")
    first = None
    for l in d["layers"]:
        s = share(f"layer{l}")
        if first is None and s >= 0.5:
            first = l
        print(f"   patched at layer {l:2d}: {s:.0%}")
    print(f"   first layer where >= 50% say the context answer: {first}")
    gen_verdicts.append(first is not None and 19 <= first <= 26)

print("\nPre-registered prediction (generation):", "HOLDS" if gen_verdicts and all(gen_verdicts) else "FAILS")

# a few examples: what the patched model actually says
d = json.load(open(sorted(glob.glob(f"{OUT_DIR}/*_generation.json"))[0]))
for i, r in list(d["items"].items())[:3]:
    print(f"\n{i}  par={items[i]['parametric_answer']}  cf={items[i]['counterfactual_answer']}")
    for cond in ["source", "target", "layer18", "layer22", "layer26"]:
        print(f"   {cond:8s} -> {r[cond]['text'][:40]!r} ({r[cond]['label']})")

# Necessity: corrupt the healthy model with the trough

Sufficiency (above) put the source's residual stream into the trough and restored context-following.
Here the direction is reversed: the **trough's** residual stream at layer `l` is put into the **source**
model (peak, or recovery), which runs the remaining layers with its own weights. If the healthy model then
answers from memory, the trough's computation up to layer `l` is enough to *cause* the flip.

Same flipping items as the sufficiency run (read from its result files), same measure:

    damage N(l) = (D_patched - D_healthy) / (D_trough - D_healthy)     0 = unchanged, 1 = fully trough-like

plus generation: share of items where the corrupted healthy model *says its own (parametric) answer*.

**Pre-registered prediction.** The first layer where median N >= 0.5 is within +-2 layers of the sufficiency
restoration layer (tulu 8, alpaca 12, tulu recovery 9), in all three comparisons, and generation follows.
Fails if the necessity curve rises elsewhere or never reaches 0.5.

In [ ]:
# Reverse jobs: healthy model = target, trough = source; items = exactly the sufficiency run's items.
LAYERS = list(range(1, 32))
GEN_LAYERS = [2, 4, 6, 8, 10, 12, 14, 16, 19, 22, 26, 31]
SUFF_RESTORATION = {("tulu", "peak"): 8, ("alpaca", "peak"): 12, ("tulu", "recovery"): 9}

NEC_JOBS = []
for job in JOBS:
    suff = json.load(open(f"{OUT_DIR}/{job['run']}_seed0_{job['source']}_to_{job['target']}.json"))
    NEC_JOBS.append({"run": job["run"], "healthy": job["source"], "items": list(suff["items"]),
                     "source_path": job["target_path"],      # trough (corrupting)
                     "target_path": job["source_path"],      # healthy model being corrupted
                     "suff": suff["items"]})
    print(f"{job['run']}: trough -> {job['source']}  on {len(suff['items'])} items")

In [ ]:
# Consistency check before the full run: with roles reversed, the unpatched healthy and trough leans must equal
# what the sufficiency run recorded for the same items (same adapters, same prompts).
job = NEC_JOBS[0]
model = load_pair(job["source_path"], job["target_path"])
worst, n1, n31 = 0.0, [], []
for i in job["items"][:6]:
    it, ids = items[i], prompt_ids(items[i])
    model.set_adapter("source"); trough = run(model, ids)
    model.set_adapter("target"); healthy = run(model, ids)
    D_tr, D_he = lean(trough.logits, it), lean(healthy.logits, it)
    worst = max(worst, abs(D_he - job["suff"][i]["D_source"]), abs(D_tr - job["suff"][i]["D_target"]))
    n1.append((lean(run(model, ids, 1, trough.hidden_states[1]).logits, it) - D_he) / (D_tr - D_he))
    n31.append((lean(run(model, ids, 31, trough.hidden_states[31]).logits, it) - D_he) / (D_tr - D_he))
print(f"leans match the sufficiency run: max difference {worst:.4f}   (want ~0)")
print(f"damage at layer 1: median {np.median(n1):+.2f}   at layer 31: median {np.median(n31):+.2f}   (want ~0 and near 1)")

In [ ]:
# Full necessity run: logit sweep over layers 1..31 plus generation at GEN_LAYERS. Resumable.
for job in NEC_JOBS:
    name = f"{job['run']}_seed0_trough_to_{job['healthy']}_necessity"
    out_file = f"{OUT_DIR}/{name}.json"
    if os.path.exists(out_file):
        print("skip (done):", out_file)
        continue
    t0 = time.time()
    model = load_pair(job["source_path"], job["target_path"])
    rows = {}
    for i in job["items"]:
        it, ids = items[i], prompt_ids(items[i])
        model.set_adapter("source")
        trough = run(model, ids)
        texts = {"trough": generate(model, ids)}
        model.set_adapter("target")
        healthy = run(model, ids)
        texts["healthy"] = generate(model, ids)
        for l in GEN_LAYERS:
            texts[f"layer{l}"] = generate(model, ids, l, trough.hidden_states[l])
        rows[i] = {"D_trough": lean(trough.logits, it), "D_healthy": lean(healthy.logits, it),
                   "D_patched": [lean(run(model, ids, l, trough.hidden_states[l]).logits, it) for l in LAYERS],
                   "gen": {k: {"text": t, "label": answer_label(t, it)} for k, t in texts.items()}}
    json.dump({"run": job["run"], "healthy": job["healthy"], "layers": LAYERS, "gen_layers": GEN_LAYERS,
               "items": rows}, open(out_file, "w"))
    print(f"{name}: {len(rows)} items in {time.time() - t0:.0f}s")

shutil.rmtree(STAGE, ignore_errors=True)

In [ ]:
# Necessity results next to sufficiency.
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), sharey=True)
nec_verdicts = []
for ax, f in zip(axes, sorted(glob.glob(f"{OUT_DIR}/*_necessity.json"))):
    d = json.load(open(f))
    rows = list(d["items"].values())
    L = np.array(d["layers"])
    N = np.array([(np.array(r["D_patched"]) - r["D_healthy"]) / (r["D_trough"] - r["D_healthy"]) for r in rows])
    med = np.median(N, axis=0)
    first = int(L[np.argmax(med >= 0.5)]) if (med >= 0.5).any() else None
    expected = SUFF_RESTORATION[(d["run"], d["healthy"])]

    suff = json.load(open(f"{OUT_DIR}/{d['run']}_seed0_{d['healthy']}_to_trough.json"))
    R = np.array([(np.array(v["D_patched"]) - v["D_target"]) / (v["D_source"] - v["D_target"]) for v in suff["items"].values()])

    ax.plot(L, np.median(R, axis=0), linewidth=2, label="sufficiency: restore trough")
    ax.plot(L, med, linewidth=2, label="necessity: corrupt healthy model")
    ax.axhline(0, color="gray", linewidth=0.8); ax.axhline(1, color="gray", linewidth=0.8, linestyle=":")
    ax.set_title(f"{d['run']}: trough <-> {d['healthy']} (n={len(rows)})"); ax.set_xlabel("patched layer")
    ax.legend(frameon=False, fontsize=8)

    own = lambda cond: np.mean([r["gen"][cond]["label"] == "PAR" for r in rows])
    print(f"\n{d['run']} trough -> {d['healthy']} (n={len(rows)}): first layer with median damage >= 0.5 = {first}"
          f"   (sufficiency restoration layer {expected})")
    print(f"   generation: unpatched {d['healthy']} says own answer {own('healthy'):.0%}, unpatched trough {own('trough'):.0%}")
    for l in d["gen_layers"]:
        k = list(L).index(l)
        print(f"   layer {l:2d}: median damage {med[k]:+.2f}   corrupted model says its own answer {own(f'layer{l}'):.0%}")
    nec_verdicts.append(first is not None and abs(first - expected) <= 2)
axes[0].set_ylabel("median fraction (0 = unchanged, 1 = other model)")
fig.tight_layout(); plt.show()

print("\nPre-registered necessity prediction (within +-2 layers of sufficiency, all three):",
      "HOLDS" if nec_verdicts and all(nec_verdicts) else "FAILS")